# Homes and the School Bus
### NYC affordable housing completions × school-bus breakdowns and delays
**Data Engineering and Visualization — Winter 2026 · Project Milestone 1**

| | |
|---|---|
| Housing & Development dataset | [Affordable Housing Production by Building](https://data.cityofnewyork.us/d/hg8x-zxpr) (`hg8x-zxpr`, catalog H2) |
| Transportation dataset | [Bus Breakdown and Delays](https://data.cityofnewyork.us/d/ez4e-fazm) (`ez4e-fazm`, catalog T5) |
| Integration key | Borough × school year (main) and borough × month (time-series / lag analysis) |
| Website | `<deployment link>` · Repository: `<GitHub link>` |

## Team and contributions

| Member (ID) | Research questions | Visualizations | Data cleaning | Integration | Website |
|---|---|---|---|---|---|
| `<Member 1 name, ID>` | RQ1, RQ2 | Scatter, two-panel time series | Housing: duplicates, borough, dates | Monthly grid + join | Search mode |
| `<Member 2 name, ID>` | RQ3, RQ4 | Bubble scatter, lag heatmap | Bus: delay-text parsing, outliers | Lagged correlation | Generate Report callback |
| `<Member 3 name, ID>` | RQ5, RQ6 | Map, grouped bar | Housing: coordinates, unit consistency | School-year aggregation | Map + layout/CSS |
| `<Member 4 name, ID>` | RQ7, RQ8 | Scatter, box plot | Bus: borough mapping, duplicates, students | Post-integration cleaning | Story section + deployment |

*Edit this table so it matches who actually did what — the same table appears in the README.*

## The question

When New York finishes an affordable building, households move in — many with children who must get to
school. The Department of Education's Office of Pupil Transportation runs school buses; HPD finances the housing.
The two agencies plan separately. **Do the boroughs and periods that deliver the most affordable housing also see
more — or longer — school-bus disruptions, and are the lowest-income and youngest riders affected differently?**

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("src").resolve()))
import pipeline as P      # all cleaning / integration functions (shared with the website)
import figures as F       # one figure function per research question (shared with the website)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e7e6e2"})

RAW = Path("data/raw"); PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Loading the data

Both datasets are downloaded through the NYC Open Data SODA API (Listing 2 in the brief). Two details matter:

* The API returns only **1,000 rows** unless `$limit` is set, so `$limit` is set above the size of both datasets.
* `$order=:id` makes the download order stable, so re-running the notebook gives identical results.

**No sampling.** The housing file has a few thousand buildings, and the bus file a few hundred thousand incidents
(exact counts are printed below). Both fit comfortably in memory, so we use the full data rather than a sample —
sampling would only throw away borough-months and make the monthly series noisier. If a Colab session runs out of
memory, set `BUS_WHERE` to a date filter such as `"occurred_on >= '2018-07-01T00:00:00'"` and note it here.

The raw downloads are cached in `data/raw/` so the notebook does not re-download on every run.

In [ ]:
BUS_WHERE = None   # e.g. "occurred_on >= '2018-07-01T00:00:00'" if memory is tight

housing_raw = P.load_dataset(P.HOUSING_ID, RAW / "housing_raw.csv")
bus_raw = P.load_dataset(P.BUS_ID, RAW / "bus_raw.csv", where=BUS_WHERE)

print(f"Housing: {housing_raw.shape[0]:,} rows × {housing_raw.shape[1]} columns")
print(f"Bus:     {bus_raw.shape[0]:,} rows × {bus_raw.shape[1]} columns")

API column names are lower-case with underscores, while the CSV export uses `Title Case With Spaces`.
`P.normalize_columns` converts both to the same snake_case names (e.g. `1-BR Units` → `br1_units`), so the rest of
the code works whichever way the data was downloaded.

In [ ]:
housing_n = P.normalize_columns(housing_raw)
bus_n = P.normalize_columns(bus_raw)
print("Housing columns:", list(housing_n.columns), "\n")
print("Bus columns:", list(bus_n.columns))

## 2. Dataset overview and why these two

| | Affordable Housing Production by Building (H2) | Bus Breakdown and Delays (T5) |
|---|---|---|
| Unit of observation | One building in an HPD-financed affordable project | One school-bus breakdown or late run reported by a contractor |
| Columns | 40+ (meets the ≥ 9 requirement) | 21 (meets the ≥ 9 requirement) |
| Geography | Borough, ZIP, community board, census tract, NTA, latitude/longitude | **Borough only** (`Boro`), plus free-text route and school codes |
| Time | Project start, project completion, building completion dates | `Occurred_On` (date-time), `School_Year` |
| Known issues | Confidential projects have no address / building id; buildings still in progress have no completion date; unit counts by income band may not sum to the total | `Boro` includes places outside NYC (Nassau, Westchester, New Jersey…); `How_Long_Delayed` is free text (`"30MINS"`, `"1hr"`, `"20"`…); edited incidents can appear more than once; no service in summer and during the 2020 COVID-19 closure |

**Why this pair.** Housing completions are a direct, dated measure of *new households arriving* in a borough;
school-bus incidents measure how well one part of the transport system is coping. Both cover the same boroughs and
overlapping years (bus data from the 2015-16 school year on), so they can be compared over time inside each
borough.

**How they will be integrated.** The bus data only knows the borough, so finer geography (ZIP, tract,
coordinates) cannot be matched — any join must be at **borough** level. In time, the bus data is organised by
**school year** (July → June), which is also the period over which new students would enrol, so:

* **Main level — borough × school year** (5 boroughs × ~9–10 school years): used for the cross-sectional questions
  (RQ1, RQ3, RQ6, RQ7, RQ8).
* **Secondary level — borough × month**: used for the time-series and lag questions (RQ2, RQ4), where a coarser
  level would hide the timing.

Both datasets are *aggregated* to these levels before joining. A row-level join would be meaningless here: there
is no link between a particular building and a particular bus incident, and the two files have very different
numbers of rows.

## 3. Exploratory data analysis
### 3.1 Housing

In [ ]:
housing_n.info(memory_usage="deep")

In [ ]:
def missing_report(df):
    out = pd.DataFrame({"missing": df.isna().sum(), "missing_%": (100 * df.isna().mean()).round(1),
                        "unique": df.nunique()})
    return out[out["missing"] > 0].sort_values("missing_%", ascending=False)

missing_report(housing_n)

In [ ]:
num_cols = ["total_units", "extremely_low_income_units", "very_low_income_units", "low_income_units",
            "br2_units", "br3_units", "latitude", "longitude"]
housing_n[[c for c in num_cols if c in housing_n]].apply(pd.to_numeric, errors="coerce").describe().T

In [ ]:
for col in ["borough", "reporting_construction_type", "extended_affordability_only", "prevailing_wage_status"]:
    print(housing_n[col].value_counts(dropna=False).to_string(), "\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
units = pd.to_numeric(housing_n["total_units"], errors="coerce")
axes[0].hist(units.dropna(), bins=np.logspace(0, np.log10(max(units.max(), 10)), 40), color="#2a78d6")
axes[0].set_xscale("log"); axes[0].set_title("Units per building (log scale)"); axes[0].set_xlabel("Total units")

done = pd.to_datetime(housing_n["building_completion_date"], errors="coerce")
by_year = housing_n.assign(year=done.dt.year).dropna(subset=["year"]).groupby(["year", "borough"]).size().unstack(fill_value=0)
by_year.plot(kind="bar", stacked=True, ax=axes[1], width=0.8, color=["#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#e34948"])
axes[1].set_title("Buildings completed per year by borough"); axes[1].set_xlabel("Completion year")
plt.tight_layout(); plt.show()

**What we see (update with your own numbers after running):** unit counts are heavily right-skewed — most buildings
are small, a few are very large towers — so we use medians and log scales rather than means. Completion dates are
missing for buildings still under construction. Latitude/longitude are missing for confidential projects.

### 3.2 Bus breakdowns and delays

In [ ]:
bus_n.info(memory_usage="deep")

In [ ]:
missing_report(bus_n)

In [ ]:
for col in ["boro", "breakdown_or_running_late", "reason", "school_age_or_prek", "run_type"]:
    print(bus_n[col].value_counts(dropna=False).head(15).to_string(), "\n")

In [ ]:
print("Most frequent raw How_Long_Delayed values:")
print(bus_n["how_long_delayed"].value_counts(dropna=False).head(25).to_string())
print("\nDistinct raw values:", bus_n["how_long_delayed"].nunique())

In [ ]:
occ = pd.to_datetime(bus_n["occurred_on"], errors="coerce")
monthly_counts = occ.dt.to_period("M").value_counts().sort_index()
ax = monthly_counts.plot(figsize=(13, 3.5), color="#2a78d6")
ax.set_title("School-bus incidents per month (all boroughs, raw)"); ax.set_ylabel("Incidents")
plt.show()

students = pd.to_numeric(bus_n["number_of_students_on_the_bus"], errors="coerce")
print(students.describe())

**What we see:** `Boro` contains values outside the five boroughs; `How_Long_Delayed` is free text with many
spellings of the same duration; incidents drop to almost zero every July–August and from March 2020 (school
closures), so those months are *no service*, not *no problems*; and the student count has impossible maxima.

## 4. Pre-integration cleaning

All steps live in `src/pipeline.py` (`clean_housing`, `clean_bus`) so the website uses exactly the same logic.
Every step is recorded in a `CleaningLog`, shown at the end of this section. Below, each decision is justified and
the evidence for it is shown before the function is run.

### 4.1 Housing — decisions

| Issue | Decision | Why |
|---|---|---|
| Exact duplicate rows; same `(project_id, building_id)` twice | Drop, keep the last | The same building would otherwise be counted twice. Confidential rows have no `building_id`, so they are only de-duplicated when the whole row is identical. |
| Borough missing / unrecognised | **Drop** | Borough is the join key; such a row cannot be placed. Very few rows. |
| `building_completion_date` missing but `project_completion_date` present | **Impute** from the project date | Buildings in a project are delivered together; this keeps real completions in the analysis. Flagged in `completion_imputed`. |
| No completion date at all | **Drop from the integrated analysis** | The building is not finished, so it has no month in which households arrived. Imputing a date would invent an event. |
| Missing unit count in an income/bedroom category | **Fill with 0** | A blank category in this file means "no units of that type". |
| Coordinates outside NYC's bounding box | **Blank** them (keep the row) | They are entry errors; the row still has a valid borough, so it stays in every borough-level result and is only left off the map. |
| Completion before start, completion in the future, `total_units ≤ 0` | **Drop** | Logically impossible values. |
| Income bands not adding up to `total_units` | **Flag** (`units_mismatch`), keep | We cannot tell which number is wrong; totals are still used. |
| Very large buildings (IQR outliers in `total_units`) | **Flag** (`units_outlier_iqr`), keep | They are real towers, not errors. Dropping them would remove exactly the developments that bring the most households. |
| Borough spelling, dates, ZIP codes, category text | Standardise | Upper-case official borough names, real datetimes, 5-digit ZIP strings, title-cased categories. |

In [ ]:
dup_exact = housing_n.duplicated().sum()
dup_key = housing_n[housing_n["building_id"].notna()].duplicated(["project_id", "building_id"]).sum()
print(f"Exact duplicate rows: {dup_exact:,}   duplicate (project_id, building_id): {dup_key:,}")

units = pd.to_numeric(housing_n["total_units"], errors="coerce")
lo, hi = P.iqr_bounds(units.dropna())
print(f"total_units IQR fences: [{lo:.0f}, {hi:.0f}] -> {(units > hi).sum():,} buildings above the upper fence "
      f"(max = {units.max():,.0f} units). These are kept and flagged.")

### 4.2 Bus — decisions

| Issue | Decision | Why |
|---|---|---|
| Exact duplicates; same `busbreakdown_id` several times | Drop; keep the **latest** `last_updated_on` | Contractors edit incidents; the last edit is the final record. |
| `Boro` outside NYC or missing | **Drop** | No NYC housing counterpart; these incidents cannot be integrated. |
| `occurred_on` missing, before July 2015 or in the future | **Drop** | No usable date to aggregate on. |
| `school_year` disagreeing with `occurred_on` | **Recompute** from the date | The date is the primary field; the label is derived from it. |
| `How_Long_Delayed` free text | **Parse** to `delay_minutes` (`P.parse_delay_minutes`) | Handles `"25 minutes"`, `"30MINS"`, `"20"`, `"1hr"`, `"1:30"`, `"45-60 min"` (mid-point) … |
| Delay unparseable, ≤ 0 or > 240 min | **Set to NaN — do not impute** | A filled-in delay would be invented data and would pull every median toward the fill value. Counts of incidents still use these rows. |
| Students on bus > 90 | **Set to NaN** | Domain rule: the largest NYC school buses seat roughly 66–90 pupils. |
| Yes/No columns, category text, company names | Standardise | Booleans; title case; collapsed whitespace. |

In [ ]:
demo = bus_n["how_long_delayed"].dropna().drop_duplicates().sample(min(15, bus_n["how_long_delayed"].nunique()), random_state=0)
pd.DataFrame({"raw text": demo, "parsed minutes": demo.map(P.parse_delay_minutes)})

In [ ]:
parsed = bus_n["how_long_delayed"].map(P.parse_delay_minutes)
lo, hi = P.iqr_bounds(parsed.dropna())
print(f"Delay (min) IQR fences: [{lo:.0f}, {hi:.0f}]")
print(f"Above IQR fence: {(parsed > hi).sum():,}   above domain cap of {P.MAX_PLAUSIBLE_DELAY_MIN} min: "
      f"{(parsed > P.MAX_PLAUSIBLE_DELAY_MIN).sum():,}   unparseable: {parsed.isna().sum():,}")

**Why a domain cap instead of the IQR fence for delays?** The IQR fence on delay minutes is low (delays cluster
around 15–45 min), so it would discard genuine one- and two-hour delays — the very cases parents care about. We
therefore *report* the IQR fence but cut only values that cannot be a delay (≤ 0, or more than 4 hours, which is a
cancelled run or a typo). The same reasoning applies to students on the bus, capped by vehicle capacity.

In [ ]:
log = P.CleaningLog()
housing = P.clean_housing(housing_raw, log)
bus = P.clean_bus(bus_raw, log)
log.to_frame()

In [ ]:
# Validation: the cleaned data no longer has the problems found in EDA.
assert housing["borough"].isin(P.BOROUGHS).all() and bus["borough"].isin(P.BOROUGHS).all()
assert housing.duplicated().sum() == 0 and bus["busbreakdown_id"].is_unique
assert housing["building_completion_date"].notna().all() and bus["occurred_on"].notna().all()
assert (bus["delay_minutes"].dropna().between(0, P.MAX_PLAUSIBLE_DELAY_MIN)).all()
assert (bus["number_of_students_on_the_bus"].dropna() <= P.MAX_STUDENTS_ON_BUS).all()
print("All pre-integration checks passed.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bus["delay_minutes"].plot(kind="hist", bins=48, ax=axes[0], color="#2a78d6")
axes[0].set_title("Delay after parsing and domain rule (minutes)")
bus["borough"].value_counts().plot(kind="barh", ax=axes[1], color="#2a78d6")
axes[1].set_title("Bus incidents per borough after removing non-NYC rows")
plt.tight_layout(); plt.show()

## 5. Integration

**Join key and granularity.** Alternatives considered:

* *ZIP code / census tract / coordinates* — the bus file has none of them, so impossible.
* *School or route* — the bus file names schools by code, but the housing file has no link to schools; matching
  buildings to schools would need a third dataset (school zones), which is outside the brief.
* *Borough × calendar year* — workable, but splits each school year across two calendar years.
* **Borough × school year** (chosen as the main level) — matches how the bus system is organised and when
  households' children would enrol.
* **Borough × month** (chosen for timing questions) — fine enough to see seasonality, the 2020 closure and lags.

**Aggregation.** Housing → buildings, units, deeply affordable units (extremely + very low income), family-sized
units (2+ bedrooms), new-construction vs. preservation units per borough-period. Bus → incidents, breakdowns,
heavy-traffic incidents, Pre-K incidents, median delay, average students per borough-period.

**Join type.** A complete borough × period *grid* is built first and both aggregates are left-joined onto it. An
inner join would silently drop borough-months with zero completions; an outer join without the grid could still
miss periods present in neither table. The window is restricted to the period both datasets cover.

In [ ]:
merged_monthly = P.integrate(housing, bus, "month")
merged_yearly = P.integrate(housing, bus, "school_year")
print("Monthly:", merged_monthly.shape, "  span:", merged_monthly["month"].min().date(), "->", merged_monthly["month"].max().date())
print("Yearly: ", merged_yearly.shape, "  school years:", merged_yearly["school_year"].min(), "->", merged_yearly["school_year"].max())
merged_monthly.head()

## 6. Post-integration cleaning

The join itself creates three kinds of problems, each inspected below and then fixed by
`P.post_integration_clean`:

1. **New missing values.** Housing NaN = no building completed that period → **0**. Bus NaN in a month when the
   whole city has (almost) no incidents = **no school-bus service** (summer, COVID closure) → kept as **NaN** and
   flagged `school_in_session = False`, so these months are excluded from averages and correlations rather than
   counted as perfect months. A borough with no incidents in an in-session month gets 0.
2. **Redundant / conflicting columns.** The monthly join carries two `school_year` columns (one derived from the
   grid month, one from the bus file). We check they agree, then keep the date-derived one. We also add
   `other_construction_units` so that construction-type units add up to the total.
3. **Data-type mismatches.** Left joins turn integer counts into floats (because of NaN); they are cast back to
   nullable integers (`Int64`). `month` is forced to datetime and `borough`/`school_year` to string so both
   aggregates use identical key types.

In [ ]:
print("Missing values created by the join (monthly):")
print(merged_monthly.isna().sum()[lambda s: s > 0].to_string())
print("\nConflicting columns:", [c for c in merged_monthly.columns if c.endswith("_bus")])
print("\nDtypes before:", merged_monthly[["units_completed", "incidents", "breakdowns"]].dtypes.to_dict())

In [ ]:
post_log = P.CleaningLog()
monthly = P.post_integration_clean(merged_monthly, "month", post_log)
yearly = P.post_integration_clean(merged_yearly, "school_year", post_log)
display(post_log.to_frame())

print("Dtypes after:", monthly[["units_completed", "incidents", "breakdowns"]].dtypes.to_dict())
print("Remaining NaN in housing counts:", int(monthly[P.HOUSING_COUNT_COLS].isna().sum().sum()))
print("Months flagged without school service:", int((~monthly["school_in_session"]).sum()))

assert not monthly.duplicated(["borough", "month"]).any() and not yearly.duplicated(["borough", "school_year"]).any()
assert monthly[P.HOUSING_COUNT_COLS].notna().all().all()
assert monthly.loc[monthly["school_in_session"], "incidents"].notna().all()
print("All post-integration checks passed.")
yearly.head(10)

### Save the processed data for the website

The website re-aggregates from the cleaned *row-level* tables (so filters like "Breakdown only" or
"New Construction only" can be applied before aggregation). Only the columns the website needs are saved,
compressed, so the repository stays small.

In [ ]:
HOUSING_APP_COLS = ["project_id", "project_name", "building_id", "borough", "postcode", "latitude", "longitude",
                    "building_completion_date", "completion_month", "school_year", "reporting_construction_type",
                    "total_units", "extremely_low_income_units", "very_low_income_units", "deeply_affordable_units",
                    "family_units"]
BUS_APP_COLS = ["busbreakdown_id", "borough", "school_year", "occurred_on", "month", "reason",
                "breakdown_or_running_late", "school_age_or_prek", "delay_minutes",
                "number_of_students_on_the_bus", "is_breakdown", "is_heavy_traffic", "is_prek"]

housing[HOUSING_APP_COLS].to_csv(PROCESSED / "housing_clean.csv.gz", index=False)
bus[BUS_APP_COLS].to_csv(PROCESSED / "bus_clean.csv.gz", index=False)
monthly.to_csv(PROCESSED / "integrated_borough_month.csv", index=False)
yearly.to_csv(PROCESSED / "integrated_borough_school_year.csv", index=False)
for f in sorted(PROCESSED.iterdir()):
    print(f"{f.name:40s} {f.stat().st_size / 1e6:6.2f} MB")

## 7. Research questions

Each member contributes two questions, each answered by one chart. The charts are produced by `src/figures.py` —
the same functions the website calls — so what you see here is what the website shows with no filters applied.

> **Interpretation cells:** the *Answer* cell under each chart must be written by the member after running the
> notebook on the real data. Describe what the chart shows, give the number (r, share, median), and say how
> confident you are. Do not claim causation.

### RQ1 (Member 1)
**Question:** Do borough-years that delivered more affordable units also log more school-bus incidents?

**Chart:** Scatter of units completed vs. incidents, one point per borough × school year, with a pooled linear fit.

In [ ]:
F.rq1_units_vs_incidents(yearly).show()

In [ ]:
r_units = P.ols_line(yearly['units_completed'], yearly['incidents'])[2]
print(f'Pearson r (all borough-years): {r_units:+.2f}')
for b, g in yearly.groupby('borough'):
    print(f'  within {b:14s} r = {P.ols_line(g["units_completed"], g["incidents"])[2]:+.2f}')

**Answer (Member 1):** *[Write after running: what the chart shows, the key number, and how confident you are.]*

**Caveat:** Pooling boroughs mixes *between*-borough size differences with *within*-borough change; the per-borough r values printed above separate the two.

### RQ2 (Member 1)
**Question:** Over time, does the 12-month pipeline of new affordable units move with monthly school-bus incidents in each borough?

**Chart:** Two panels on a shared time axis (rolling 12-month completions; monthly incidents with gaps where there is no service). Two panels instead of a dual-axis chart, so neither scale is distorted.

In [ ]:
F.rq2_timeline(monthly).show()

**Answer (Member 1):** *[Write after running: what the chart shows, the key number, and how confident you are.]*

**Caveat:** Look for whether peaks in the housing panel are followed by rises in the bus panel, and how the 2020 closure breaks the series.

### RQ3 (Member 2)
**Question:** Where a larger share of new units is reserved for extremely and very low-income households, are school-bus delays longer?

**Chart:** Bubble scatter: x = % of units that are extremely/very low income, y = median reported delay, bubble size = units completed.

In [ ]:
F.rq3_deep_affordability_vs_delay(yearly).show()

**Answer (Member 2):** *[Write after running: what the chart shows, the key number, and how confident you are.]*

**Caveat:** Median delays are reported in coarse steps (15, 20, 30 min…), so small differences may be reporting artefacts.

### RQ4 (Member 2)
**Question:** Do housing completions precede school-bus incidents, and by how many months?

**Chart:** Heatmap of Pearson r between completions shifted by 0–12 months and monthly incidents, per borough (diverging scale, grey = no correlation).

In [ ]:
F.rq4_lag_heatmap(monthly).show()

In [ ]:
lags = P.lagged_correlations(monthly)
lags.pivot(index='borough', columns='lag_months', values='r').round(2)

**Answer (Member 2):** *[Write after running: what the chart shows, the key number, and how confident you are.]*

**Caveat:** Both series share the school-year rhythm; a positive lag correlation can come from that common seasonality alone.

### RQ5 (Member 3)
**Question:** Is new affordable housing concentrated in the same boroughs where school-bus disruptions concentrate?

**Chart:** Map: each building as a dot (size = units, colour = new construction vs. preservation) over grey borough bubbles sized by total bus incidents.

In [ ]:
F.rq5_map(housing, yearly).show()

In [ ]:
share = pd.DataFrame({'% of units': 100 * housing.groupby('borough')['total_units'].sum() / housing['total_units'].sum(),
                      '% of bus incidents': 100 * bus.groupby('borough').size() / len(bus)}).round(1)
share

**Answer (Member 3):** *[Write after running: what the chart shows, the key number, and how confident you are.]*

**Caveat:** Bus bubbles sit at borough centres because the bus data has no finer location.

### RQ6 (Member 3)
**Question:** In a borough's high new-construction years, is heavy traffic a larger share of delay reasons than in its low new-construction years?

**Chart:** Grouped bar per borough: heavy-traffic share in years above vs. below that borough's own median of new-construction units.

In [ ]:
F.rq6_heavy_traffic_share(yearly).show()

**Answer (Member 3):** *[Write after running: what the chart shows, the key number, and how confident you are.]*

**Caveat:** Each borough is compared only with itself, so differences in borough size do not drive the result.

### RQ7 (Member 4)
**Question:** Do borough-years that add more family-sized units (2+ bedrooms) see more students on board when a bus is disrupted?

**Chart:** Scatter of family-sized units vs. average students on the bus per incident, per borough × school year.

In [ ]:
F.rq7_family_units_vs_students(yearly).show()

**Answer (Member 4):** *[Write after running: what the chart shows, the key number, and how confident you are.]*

**Caveat:** The student count is often recorded as 0 or left blank, which pulls averages down.

### RQ8 (Member 4)
**Question:** Are Pre-K and school-age riders delayed for different lengths of time in high- vs. low-completion years?

**Chart:** Box plot of delay minutes by rider group, split into each borough's above- and below-median completion years.

In [ ]:
F.rq8_prek_delay_box(bus, yearly).show()

In [ ]:
rider = np.where(bus['is_prek'], 'Pre-K', 'School-age')
bus.groupby(rider)['delay_minutes'].describe().round(1)

**Answer (Member 4):** *[Write after running: what the chart shows, the key number, and how confident you are.]*

**Caveat:** Delay minutes are self-reported by contractors and rounded.

## 8. Limitations

* **Correlation is not causation.** Housing completions and bus incidents both follow citywide trends (enrolment,
  contractor changes, the COVID-19 closure) that neither dataset records.
* **Borough-level only.** The bus data has no finer location, so a building and a delayed bus counted in the same
  borough may be far apart. Five boroughs × ~10 school years is a small sample for correlations.
* **Partial housing picture.** Only HPD-financed affordable units are included, not market-rate construction.
* **No denominator for buses.** The bus file gives no number of routes or riders, so incident counts are compared
  within each borough over time rather than across boroughs.
* **Self-reported delay text.** Unparseable or implausible delays were left blank, not imputed.

## 9. Who could use this

* **DOE Office of Pupil Transportation** — anticipate route capacity in boroughs with large upcoming completions.
* **HPD** — share the completion pipeline with OPT before buildings open.
* **Families and advocates** — evidence for requests about bus service near new affordable housing.

## 10. Prompts used

The brief asks for every prompt used. Add each prompt the team used with an AI assistant, with the step it was used
for.

| Step | Tool | Prompt |
|---|---|---|
| Project scaffolding (pipeline, notebook structure, website) | Claude | "i want to implement what these descriptions asked for in my data engineering and visualization course project for my uni" (with the project brief and dataset catalog attached); follow-up choices: datasets H2 + T5, 4 members, Dash on Render, full project |
| | | |

## 11. Dataset links

* Affordable Housing Production by Building — https://data.cityofnewyork.us/d/hg8x-zxpr
* Bus Breakdown and Delays — https://data.cityofnewyork.us/d/ez4e-fazm